# Bounding the loop: budgets, caps, no-progress

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 16 §16.6–16.8 · type: walkthrough*

**The promise:** make termination a *provable* property of an agent run — step, token, dollar, and wall-clock
caps plus repeated-action detection — enforced by **code**, never the prompt.


## 🧠 Why this matters

Every pattern in this chapter contains a loop, and every loop where **the model decides whether to continue**
has *no intrinsic termination guarantee*. The model is probabilistic: some fraction of runs will retry a failing
tool forever, "verify" in circles, or chase a goal that no longer exists. At scale, that fraction meets your
invoice.

This is the notebook the chapter insists on hardest. Termination is **architectural**, not a prompt property.
"This agent cannot spend more than \$0.40 or 60 seconds per request, *by construction*" is what makes an agent
deployable. We build the `RunBudget` from §16.7, wire it into the 16-01 loop, watch it trip, and **fail well**.


## Objectives & prerequisites

**By the end you can:**
- Build `RunBudget` with `charge()` + `raise_if_spent()` and four caps (steps, tokens, seconds, repeats).
- Detect the most common runaway signature: **same tool + same hashed args** repeated.
- Drop an enforced budget into any loop in this Part, and **fail well** on a trip (checkpoint + partial result + metrics).
- Apply the §16.6 context-engineering moves and walk the §16.8 failure-mode taxonomy as a checklist.

**Run first / prereqs:**
- `16-01` (a loop to bound) · `16-02` (patterns that each contain a loop).
- Ch 14 (checkpoints, notes) · Ch 23 (stop-reason metrics, dashboards).
- **Offline by design** — a fake model fn makes the budget logic deterministic and free. No key needed, ever.


## Setup

This notebook is **offline by design**: the budget logic must be testable deterministically, so even the
"model" is a fake function. `MOCK=1` is the only path that matters here; the optional live path simply mirrors
16-01. We use a *fake clock* so the deadline test is reproducible (no real wall-clock flakiness in CI).


In [ ]:
import os
import time
import json
import random
import hashlib
from dataclasses import dataclass, field, asdict

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

MOCK = os.getenv("COMPANION_MOCK", "1") == "1"
MODEL = "claude-opus-4-8"
random.seed(16)

# Fake monotonic clock so the deadline cap is deterministic in CI.
class FakeClock:
    def __init__(self):
        self.t = 0.0
    def monotonic(self):
        return self.t
    def tick(self, seconds):
        self.t += seconds

CLOCK = FakeClock()
print("MOCK mode (offline by design): budget logic is deterministic and free.")


## ⚠️ The canonical incident

The classic outage: an agent retries a failing API call in a tight loop overnight. Because **each retry re-sends
the entire growing context**, cost grows *quadratically* with iterations — and the team finds it via the billing
alert, not the error logs. The model doesn't get bored, doesn't get suspicious, and doesn't see the meter.

Let's make the quadratic real before we kill it.


In [ ]:
def cost_if_unbounded(iterations: int, base_context: int = 2000, growth: int = 500) -> int:
    """Each retry re-sends the whole transcript, which grew by the last response."""
    total = 0
    ctx = base_context
    for _ in range(iterations):
        total += ctx          # you pay to re-send the whole context every turn
        ctx += growth         # ...and it keeps growing
    return total

for n in (10, 100, 1000):
    print(f"{n:>5} retries -> ~{cost_if_unbounded(n):>12,} tokens billed")
print("\nLinear iterations, QUADRATIC tokens. This is why 'it usually stops' is not a plan.")


## Build `RunBudget` (§16.7)

Make the budget an **explicit object every loop must consume from** — one mechanism, enforced everywhere.
`charge()` records a turn's spend and its action signature; `raise_if_spent()` trips on the first cap exceeded.
The four caps — steps, tokens, seconds, repeats — plus the repeated-action check are the whole guarantee.


In [ ]:
class BudgetExceeded(Exception):
    """Raised when a RunBudget cap trips. Carries the stop reason."""

@dataclass
class Usage:
    input_tokens: int = 0
    output_tokens: int = 0

class RunBudget:
    """Hard limits for one agent run. raise_if_spent() in every loop."""

    def __init__(self, max_steps=15, max_tokens=200_000,
                 max_seconds=300, max_repeats=2, clock=time):
        self.max_steps, self.max_tokens = max_steps, max_tokens
        self._clock = clock
        self.deadline = clock.monotonic() + max_seconds
        self.max_repeats = max_repeats
        self.steps = self.tokens = 0
        self.recent_actions: list[str] = []

    def charge(self, usage: Usage, action_sig: str):
        self.steps += 1
        self.tokens += usage.input_tokens + usage.output_tokens
        self.recent_actions.append(action_sig)

    def raise_if_spent(self):
        if self.steps >= self.max_steps:
            raise BudgetExceeded("step cap")
        if self.tokens >= self.max_tokens:
            raise BudgetExceeded("token cap")
        if self._clock.monotonic() >= self.deadline:
            raise BudgetExceeded("deadline")
        sig = self.recent_actions[-1:] * (self.max_repeats + 1)
        if self.recent_actions[-len(sig):] == sig and sig:
            raise BudgetExceeded("no progress: repeated action")

def action_signature(name: str, args: dict) -> str:
    """tool name + hashed arguments — the runaway fingerprint."""
    h = hashlib.sha256(json.dumps(args, sort_keys=True).encode()).hexdigest()[:8]
    return f"{name}:{h}"

print("RunBudget ready. signature example:", action_signature("search_docs", {"query": "atlas"}))


## Wire it into the 16-01 ReAct loop

Every turn must consume from the budget. We re-use the 16-01 loop shape, but now `charge()` runs before each
model call and `raise_if_spent()` gates the next iteration. The budget is external to the model — the prompt
cannot talk its way past it.


In [ ]:
@dataclass
class Block:
    type: str
    name: str = ""
    input: dict = field(default_factory=dict)
    text: str = ""

@dataclass
class Response:
    stop_reason: str
    content: list
    usage: Usage = field(default_factory=lambda: Usage(800, 120))

def looping_model(_messages):
    """A deliberately broken 'model': it calls the SAME tool with the SAME args forever."""
    return Response("tool_use", [Block("tool_use", name="search_docs", input={"query": "atlas"})])

def run_tool(name, args):
    return "No document matched that query."  # never makes progress

def bounded_react(task: str, budget: RunBudget, model_fn):
    messages = [{"role": "user", "content": task}]
    while True:
        budget.raise_if_spent()          # external guard BEFORE the spend
        resp = model_fn(messages)
        tool_blocks = [b for b in resp.content if b.type == "tool_use"]
        sig = action_signature(tool_blocks[0].name, tool_blocks[0].input) if tool_blocks else "final"
        budget.charge(resp.usage, sig)   # record spend + action signature
        if resp.stop_reason != "tool_use":
            return resp
        for b in tool_blocks:
            run_tool(b.name, b.input)
        messages.append({"role": "user", "content": "observation"})


### 🔮 Predict

We run the broken `looping_model` (same tool, same args, forever) under a budget of `max_steps=15`,
`max_repeats=2`. **Which cap fires first** — the step cap at 15, or the no-progress check? Predict before running.


In [ ]:
budget = RunBudget(max_steps=15, max_repeats=2, clock=CLOCK)
try:
    bounded_react("find the atlas lead", budget, looping_model)
except BudgetExceeded as e:
    print(f"tripped after {budget.steps} steps -> stop_reason: {e}")


**What you just saw.** **No-progress** fired at step 3, long before the step cap at 15 — three identical
signatures (`max_repeats=2` means a 3rd repeat trips it). The single most common runaway signature is an agent
calling the *same tool with the same arguments*, and it's trivially detectable by comparing action signatures.
That check earns its keep: it catches the loop in 3 steps instead of letting it burn the full 15.


## See each cap fire

Termination is only "provable" if *every* bound actually trips. Let's force each one in isolation.


In [ ]:
def trip(**kw):
    clk = FakeClock()
    b = RunBudget(clock=clk, **{k: v for k, v in kw.items() if k != 'advance'})
    try:
        for _ in range(50):
            b.raise_if_spent()
            b.charge(Usage(50_000, 50_000), action_signature("f", {"i": random.random()}))
            clk.tick(kw.get("advance", 0))
    except BudgetExceeded as e:
        return str(e), b.steps
    return "never tripped", b.steps

print("step cap   :", trip(max_steps=4, max_tokens=10**9, max_seconds=10**9))
print("token cap  :", trip(max_steps=999, max_tokens=250_000, max_seconds=10**9))
print("deadline   :", trip(max_steps=999, max_tokens=10**9, max_seconds=5, advance=2))


## Fail *well*: checkpoint, partial result, metrics

A budget trip is not a crash to swallow — it's a controlled stop. On a trip: **checkpoint** the run state (the
Ch 14 pattern), return a **partial result** with a stop-reason, and **emit metrics** (steps / tokens / dollars /
stop_reason) so Ch 23's dashboards can show you the distribution. This is what turns "it died" into "it stopped,
here's what it had, here's why."


In [ ]:
TOKEN_PRICE = 3e-6  # $/token, illustrative blended rate

@dataclass
class RunResult:
    stop_reason: str
    partial: str
    checkpoint: dict
    metrics: dict

def run_with_failwell(task, budget, model_fn, notes=None):
    notes = notes or []
    try:
        resp = bounded_react(task, budget, model_fn)
        partial, stop = "<completed>", "end_turn"
    except BudgetExceeded as e:
        partial, stop = f"partial: best-effort notes={notes}", str(e)
    metrics = {
        "steps": budget.steps,
        "tokens": budget.tokens,
        "dollars": round(budget.tokens * TOKEN_PRICE, 4),
        "stop_reason": stop,
    }
    checkpoint = {"task": task, "notes": notes, "last_actions": budget.recent_actions[-3:]}
    return RunResult(stop, partial, checkpoint, metrics)

res = run_with_failwell("find the atlas lead", RunBudget(max_repeats=2, clock=CLOCK), looping_model,
                        notes=["checked project-atlas doc"])
print("stop_reason:", res.stop_reason)
print("metrics    :", res.metrics)
print("checkpoint :", res.checkpoint)


## Context-engineering primer (§16.6): curate, don't append

Long runs fail two ways: the **hard wall** (you exhaust the window) and the subtler **context rot** (a critical
fact buried mid-window is *functionally forgotten* — "lost in the middle"). The fix is to treat the window as
**working memory you curate every turn**, not a log you append to. We demo **compaction** on a tiny synthetic
transcript (generated in-cell, not committed): fold stale turns into a brief, keep the goal and the latest
results, and place them at the **edges** where the model attends most.


In [ ]:
# Generate a tiny synthetic transcript in-cell (never committed as a blob).
GOAL = "Find where the Atlas tech lead is based."
transcript = [{"role": "assistant", "text": f"step {i}: searched, got routine result {i}"} for i in range(12)]
transcript.append({"role": "assistant", "text": "step 12: FOUND lead = Mara Quinn, office = Lisbon"})

def compact(goal, turns, keep_recent=2):
    stale, recent = turns[:-keep_recent], turns[-keep_recent:]
    brief = f"[summary of {len(stale)} earlier turns: routine searches, no decisive result]"
    # Goal at the START, recent + critical results at the END (the attended edges).
    return [f"GOAL: {goal}", brief] + [t["text"] for t in recent]

before = sum(len(t["text"]) for t in transcript)
window = compact(GOAL, transcript)
after = sum(len(s) for s in window)
print(f"raw transcript: {before} chars across {len(transcript)} turns")
print(f"curated window: {after} chars\n")
print("\n".join(window))


Goal at the top, decisive result at the bottom, stale middle compressed away. Same facts, but now they sit
where the model actually looks. Context engineering is as much about **position** as size.


## 📋 Failure-mode taxonomy checklist (§16.8)

A senior walks the taxonomy at every design review and postmortem. For each row, force two answers: **what
detects it** and **what bounds it**. A mode with neither is a latent incident. Here are three this chapter owns —
verify the detector and bound are wired in the code above.


In [ ]:
TAXONOMY = [
    {"failure": "Runaway loop / no termination",
     "detects": "step/token/deadline metric crosses cap",
     "bounds": "RunBudget hard caps (raise_if_spent)"},
    {"failure": "No-progress repeated action",
     "detects": "same tool+hashed-args signature repeats > max_repeats",
     "bounds": "action-signature repeat detection"},
    {"failure": "Premature success / give-up",
     "detects": "exit without passing the verification ladder (16-02)",
     "bounds": "require hard verification before exit"},
]

for row in TAXONOMY:
    print(f"• {row['failure']}")
    print(f"    detects: {row['detects']}")
    print(f"    bounds : {row['bounds']}")


## 🎯 Senior lens

**Termination is architectural, not a prompt property.** In a design review, ask the question you'd ask of any
distributed workflow: "what are the **provable** bounds on this run — steps, tokens, time, dollars — and what
enforces them?" If the answer involves the model's judgment, the answer is *there are none*. Bounded autonomy is
also what makes agents deployable politically: the team that can say "this agent cannot spend more than \$0.40
or 60 seconds per request, by construction" ships; the team that says "it usually stops" does not.


## Recap

- A model-driven loop has **no intrinsic termination**; unbounded retries grow cost **quadratically**.
- `RunBudget` makes termination a **provable property** via four caps + `raise_if_spent()` in every loop.
- **Repeated-action detection** (tool + hashed args) catches the most common runaway in ~3 steps, not 15.
- On a trip, **fail well**: checkpoint state, return a partial result with a stop-reason, emit metrics (→ Ch 23).
- **Context engineering** (§16.6): curate the window every turn — compact stale turns, keep goal + results at the edges.
- Walk the **failure-mode taxonomy**: every mode needs both a *detector* and a *bound*.


## Exercises

1. **Add a dollar cap.** Extend `RunBudget` with `max_dollars` (tokens × price). Predict which cap trips first on
   the looping model when dollars are tight, then verify.
2. **Defeat the signature, then re-catch it.** Make the broken model vary one trivial arg each turn so the hash
   changes. Which cap saves you now? Add a *semantic* no-progress check (no new info in the observation).
3. **Hierarchical budget (Ch 17 preview).** Give a parent budget two child budgets that draw from it. Show a
   child trip without tripping the parent, and a child trip that exhausts the parent.
4. **Real wall clock.** Swap `FakeClock` for `time` and a tiny `max_seconds`; confirm the deadline still trips.
   Why did we use a fake clock in CI?


In [ ]:
# Exercise 1: add max_dollars to RunBudget and trip it.


In [ ]:
# Exercise 2: vary an arg so the signature changes; add a semantic no-progress check.


In [ ]:
# Exercise 3: parent + child budgets (foreshadows Ch 17's tree-shaped budget).


In [ ]:
# Exercise 4: swap in the real clock and confirm the deadline cap fires.


## Next

- **Blueprint (the real one):** the `RunBudget` + termination guards seed
  [`../../blueprints/agent-loop/`](../../blueprints/agent-loop/) — the production loop hardening. You built
  the enforced budget; that's where it becomes the loop's spine.
- **Template:** the bounded-loop defaults flow into
  [`../../templates/agent-project-starter/`](../../templates/agent-project-starter/).
- **Capstone:** `RunBudget` becomes the hierarchical budget every
  [`../../capstone-project/`](../../capstone-project/) agent run consumes (Ch 17 makes it tree-shaped); checkpoint
  `checkpoints/ch16-runbudget`.
- **Next chapter:** Ch 17 (multi-agent supervisor) reuses this budget tree-shaped.
- **Book:** see §16.6–16.8.
